In [38]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Passo 1: Preparar o conjunto de treinamento
treino = pd.read_csv('treino_out_deliverytime.csv')

# Tratar valores ausentes nas variáveis numéricas
numeric_cols = treino.select_dtypes(include=['float64', 'int64']).columns
treino[numeric_cols] = treino[numeric_cols].fillna(treino[numeric_cols].mean())

# Tratar valores ausentes nas variáveis categóricas, excluindo 'order_id'
categorical_cols = treino.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
treino[categorical_cols] = treino[categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])

# Remover 'order_id' das features
treino = treino.drop(columns=['order_id'], errors='ignore')

# Passo 2: Codificar variáveis categóricas no conjunto de treinamento
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
treino[categorical_cols] = ordinal_encoder.fit_transform(treino[categorical_cols])

# Passo 3: Separar dados para prever 'approval_to_carrier_days' e 'carrier_to_customer_days'
X_intermediate = treino.drop(columns=['approval_to_carrier_days', 'carrier_to_customer_days', 'delivery_time (days)', 'freight_value', 'payment_value', 'seller_city', 'seller_state', 'product_category_name', 'product_height_cm','purchase_to_approval_days'])
y_approval = treino['approval_to_carrier_days']
y_carrier = treino['carrier_to_customer_days']

# Passo 4: Treinar modelos para 'approval_to_carrier_days' e 'carrier_to_customer_days'

# Modelo para 'approval_to_carrier_days'
model_approval = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=716,         # Aumente o número de estimadores
    learning_rate=0.011459171910607644,       # Reduza a taxa de aprendizado
    max_depth=9,              # Profundidade das árvores
    min_child_weight=9,      # Peso mínimo por folha
    subsample=0.6988466463923404,            # Subamostragem de linhas
    colsample_bytree=0.8533088440608811,     # Subamostragem de colunas
    reg_alpha=0.0009902391365606555,            # Regularização L1
    reg_lambda=0.7565432460670021,             # Regularização L2
    gamma=0.498568613419369                 # Reduzir a complexidade do modelo
)
model_approval.fit(X_intermediate, y_approval)

# Modelo para 'carrier_to_customer_days'
model_carrier = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=801,         # Aumente o número de estimadores
    learning_rate=0.018190619887727577,       # Reduza a taxa de aprendizado
    max_depth=7,              # Profundidade das árvores
    min_child_weight=10,      # Peso mínimo por folha
    subsample=0.6812398856987514,            # Subamostragem de linhas
    colsample_bytree=0.6432791404141186,     # Subamostragem de colunas
    reg_alpha=0.48691225203383853,            # Regularização L1
    reg_lambda=2.092554847706217,             # Regularização L2
    gamma=0.1956394716734175                 # Reduzir a complexidade do modelo
)
model_carrier.fit(X_intermediate, y_carrier)

# Passo 5: Preparar o conjunto de teste
teste = pd.read_csv('Teste_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
teste_numeric_cols = teste.select_dtypes(include=['float64', 'int64']).columns
teste[teste_numeric_cols] = teste[teste_numeric_cols].fillna(treino[numeric_cols].mean())  # Usando médias do treino

# Tratar valores ausentes nas variáveis categóricas
teste_categorical_cols = teste.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
teste[teste_categorical_cols] = teste[teste_categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])  # Usando modos do treino

# Remover 'order_id' das features
teste = teste.drop(columns=['order_id'], errors='ignore')

# Codificar variáveis categóricas no conjunto de teste
teste_categorical_cols = [col for col in categorical_cols if col in teste.columns]
teste[teste_categorical_cols] = ordinal_encoder.transform(teste[teste_categorical_cols])

# Garantir a mesma ordem de colunas
teste_intermediate = teste[X_intermediate.columns]

# Passo 6: Prever 'approval_to_carrier_days' e 'carrier_to_customer_days' no conjunto de teste
approval_pred = model_approval.predict(teste_intermediate)
carrier_pred = model_carrier.predict(teste_intermediate)

# Passo 7: Calcular o 'delivery_time (days)' como a soma das previsões e 'purchase_to_approval_days' do conjunto de teste
teste_final = teste_intermediate.copy()
teste_final['approval_to_carrier_days'] = approval_pred
teste_final['carrier_to_customer_days'] = carrier_pred
teste_final['purchase_to_approval_days'] = teste['purchase_to_approval_days']

# Calcular 'delivery_time (days)' somando as colunas
teste_final['delivery_time (days)'] = (
    teste_final['purchase_to_approval_days'] +
    teste_final['approval_to_carrier_days'] +
    teste_final['carrier_to_customer_days']
)

# Passo 8: Criar um DataFrame com 'order_id' e 'delivery_time (days)'
df_predictions = pd.DataFrame({
    'order_id': order_ids_test,
    'delivery_time (days)': teste_final['delivery_time (days)']
})

# Passo 9: Salvar as previsões em um arquivo CSV
df_predictions.to_csv('previsoes.csv', index=False)

# Opcional: Exibir as primeiras linhas das previsões
print(df_predictions.head())


                           order_id  delivery_time (days)
0  00024acbcdf0a6daa1e931b038114c75              8.981620
1  000576fe39319847cbb9d288c5617fa6             11.365003
2  0005f50442cb953dcd1d21e1fb923495              5.409551
3  00063b381e2406b52ad429470734ebd5             10.290338
4  0006ec9db01a64e59a68b2c340bf65a7             14.481019


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Passo 1: Preparar o conjunto de treinamento
treino = pd.read_csv('Treino_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
numeric_cols = treino.select_dtypes(include=['float64', 'int64']).columns
treino[numeric_cols] = treino[numeric_cols].fillna(treino[numeric_cols].mean())

# Tratar valores ausentes nas variáveis categóricas, excluindo 'order_id'
categorical_cols = treino.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
treino[categorical_cols] = treino[categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])

# Remover 'order_id' das features
treino = treino.drop(columns=['order_id'], errors='ignore')

# Passo 2: Codificar variáveis categóricas no conjunto de treinamento
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
treino[categorical_cols] = ordinal_encoder.fit_transform(treino[categorical_cols])

# Passo 3: Separar dados para prever 'approval_to_carrier_days' e 'carrier_to_customer_days'
X_intermediate = treino.drop(columns=['approval_to_carrier_days', 'carrier_to_customer_days', 'delivery_time (days)','freight_value'  'payment_value', 'product_weight_g'])
y_delivery_time = treino['delivery_time (days)']


# Passo 4: Treinar modelos para 'approval_to_carrier_days' e 'carrier_to_customer_days'

# Modelo para 'approval_to_carrier_days'
model_approval = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,         # Aumente o número de estimadores
    learning_rate=0.01,       # Reduza a taxa de aprendizado
    max_depth=5,              # Profundidade das árvores
    min_child_weight=10,      # Peso mínimo por folha
    subsample=0.8,            # Subamostragem de linhas
    colsample_bytree=0.8,     # Subamostragem de colunas
    reg_alpha=0.1,            # Regularização L1
    reg_lambda=1,             # Regularização L2
    gamma=0.1                 # Reduzir a complexidade do modelo
)
model_approval.fit(X_intermediate, y_delivery_time)


# Passo 5: Preparar o conjunto de teste
teste = pd.read_csv('Teste_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
teste_numeric_cols = teste.select_dtypes(include=['float64', 'int64']).columns
teste[teste_numeric_cols] = teste[teste_numeric_cols].fillna(treino[numeric_cols].mean())  # Usando médias do treino

# Tratar valores ausentes nas variáveis categóricas
teste_categorical_cols = teste.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
teste[teste_categorical_cols] = teste[teste_categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])  # Usando modos do treino

# Remover 'order_id' das features
teste = teste.drop(columns=['order_id'], errors='ignore')

# Codificar variáveis categóricas no conjunto de teste
teste_categorical_cols = [col for col in categorical_cols if col in teste.columns]
teste[teste_categorical_cols] = ordinal_encoder.transform(teste[teste_categorical_cols])

# Garantir a mesma ordem de colunas
teste_intermediate = teste[X_intermediate.columns]

# Passo 6: Prever 'approval_to_carrier_days' e 'carrier_to_customer_days' no conjunto de teste
delivery_pred = model_approval.predict(teste_intermediate)

# Passo 7: Calcular o 'delivery_time (days)' como a soma das previsões e 'purchase_to_approval_days' do conjunto de teste
teste_final = teste_intermediate.copy()
teste_final['delivery_time (days)'] = delivery_pred


# Passo 8: Criar um DataFrame com 'order_id' e 'delivery_time (days)'
df_predictions = pd.DataFrame({
    'order_id': order_ids_test,
    'delivery_time (days)': teste_final['delivery_time (days)']
})

# Passo 9: Salvar as previsões em um arquivo CSV
df_predictions.to_csv('previsoes.csv', index=False)

# Opcional: Exibir as primeiras linhas das previsões
print(df_predictions.head())


                           order_id  delivery_time (days)
0  00024acbcdf0a6daa1e931b038114c75             10.470740
1  000576fe39319847cbb9d288c5617fa6             13.162495
2  0005f50442cb953dcd1d21e1fb923495              6.852475
3  00063b381e2406b52ad429470734ebd5             11.110459
4  0006ec9db01a64e59a68b2c340bf65a7             19.808966
